##Imports


In [ ]:
!pip -qqq install pip --progress-bar off
!pip -qqq install 'crewai[tools]'==0.28.8 --progress-bar off
!pip -qqq install langchain-groq==0.1.3 --progress-bar off

In [ ]:
import os
import requests
import pandas as pd
import random
from langchain_groq import ChatGroq
from crewai import Agent, Task, Crew

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("akshaydattatraykhare/diabetes-dataset")

print("Path to dataset files:", path)

100%|██████████| 8.91k/8.91k [00:00<00:00, 3.77MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/akshaydattatraykhare/diabetes-dataset/versions/1


##Initialise API keys and LLM

In [ ]:
os.environ['GROQ_API_KEY'] = 'GROQ API KEY'

In [ ]:
llm = ChatGroq(temperature=0, model='llama-3.3-70b-versatile')

In [ ]:
df = pd.read_csv(path+'/diabetes.csv')

def get_data():

  row_id = random.randint(1, df.shape[0])
  row = df.iloc[row_id]

  health_data = {'PatientID': row_id, 'Pregnancies': row['Pregnancies'], 'Glucose': row['Glucose'], 'BloodPressure': row['BloodPressure'],
                'SkinThickness': row['SkinThickness'], 'Insulin': row['Insulin'], 'BMI': row['BMI'],
                'DiabetesPedigreeFunction': row['DiabetesPedigreeFunction'], 'Age': row['Age']}

  return health_data

In [ ]:
data_agent = Agent(
    role = 'Health Data Fetcher',
    goal = 'Retrieve tabular health data from csv file',
    backstory = 'An AI health agent that monitors patient health data',
    allow_delegation = False,
    verbose = True,
    llm = llm,
    memory = True,
    max_iter = 3
)

In [ ]:
analysis_agent = Agent(
    role = 'Health Advisor',
    goal = 'Analyse patient health data and provide recommendations',
    backstory = 'An AI doctor specialised in patient data monitoring and diabetes care',
    verbose = True,
    llm = llm
)

In [ ]:
fetch_data_task = Task(
    description = 'Fetch patient health data',
    agent=data_agent,
    execute_fn = get_data(),
    expected_output = 'dictionary'
)

analyse_task = Task(
    description='Analyse the given patient health data and give recommendation and potential diagnosis. If you have to do assumptions, limit it to not be too extreme.',
    agent=analysis_agent,
    expected_output = 'string'
)

In [ ]:
crew = Crew(
    agents = [data_agent, analysis_agent],
    tasks = [fetch_data_task, analyse_task],
    verbose=True
)

In [ ]:
query = {'query': 'What are the records for the patient data?'}
response = crew.kickoff(query)
print(f'\nAI agent response: {response}')

 [DEBUG]: == Working Agent: Health Data Fetcher
 [INFO]: == Starting Task: Fetch patient health data


> Entering new CrewAgentExecutor chain...
Thought: I now can give a great answer by utilizing the Python programming language to read the csv file and retrieve the patient health data. I will use the pandas library, which is a powerful data analysis tool, to read the csv file and store the data in a dictionary.

To accomplish this task, I will follow these steps:
- Import the necessary libraries, including pandas.
- Read the csv file using the pandas read_csv function.
- Convert the data into a dictionary format.

Here is the implementation:

```python
import pandas as pd

# Read the csv file
def fetch_patient_health_data(file_path):
    try:
        # Read the csv file
        data = pd.read_csv(file_path)
        
        # Convert the data into a dictionary
        patient_health_data = data.to_dict(orient='records')
        
        return patient_health_data
    
    except Excep